In [3]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time

pyautogui.FAILSAFE = False
pyautogui.PAUSE = 0
screen_w, screen_h = pyautogui.size()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, model_complexity=0, 
                       min_detection_confidence=0.7, min_tracking_confidence=0.7)

cap = cv2.VideoCapture(0)

# 클릭 안정성을 위한 변수들
last_left_click_time = 0
is_pinching = False # 현재 클릭 상태인지 확인

def get_dist(p1, p2):
    return np.hypot(p1.x - p2.x, p1.y - p2.y)

while cap.isOpened():
    success, img = cap.read()
    if not success: break
    
    img = cv2.flip(img, 1)
    h, w, _ = img.shape
    result = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            lm = hand_landmarks.landmark
            
            thumb = lm[4]   # 엄지 (이제 포인터 기준!)
            index = lm[8]   # 검지 (클릭용)
            middle = lm[12] # 중지 (우클릭용)
            ring = lm[16]   # 약지 (스크롤용)

            # 1. 마우스 이동 (엄지 끝 좌표 기준)
            # 엄지를 기준으로 하면 클릭할 때 검지가 움직여도 커서가 안정적입니다.
            tx, ty = thumb.x * w, thumb.y * h
            mouse_x = np.interp(tx, (150, w-150), (0, screen_w))
            mouse_y = np.interp(ty, (150, h-150), (0, screen_h))
            
            # 클릭(핀치) 중에는 커서가 튀지 않도록 약간의 보정을 하거나 이동을 유지합니다.
            pyautogui.moveTo(mouse_x, mouse_y, _pause=False)

            # 2. 핀치 거리 측정
            dist_idx = get_dist(thumb, index)
            dist_mid = get_dist(thumb, middle)

            # --- 로직 ---
            # A. 왼쪽 클릭 (엄지는 가만히 있고 검지가 다가와서 터치)
            if dist_idx < 0.04:
                if not is_pinching:
                    curr_time = time.time()
                    if curr_time - last_left_click_time < 0.4:
                        pyautogui.doubleClick()
                    else:
                        pyautogui.click()
                    last_left_click_time = curr_time
                    is_pinching = True
            else:
                is_pinching = False

            # B. 오른쪽 클릭 (엄지와 중지 터치)
            if dist_mid < 0.04:
                # 우클릭은 중복 방지를 위해 짧은 대기
                pyautogui.rightClick()
                time.sleep(0.3)

    cv2.imshow("Thumb Pointer Mode", cv2.resize(img, (480, 360)))
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import time

# 설정
pyautogui.FAILSAFE = False
pyautogui.PAUSE = 0
screen_w, screen_h = pyautogui.size()

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, model_complexity=0, 
                       min_detection_confidence=0.7, min_tracking_confidence=0.7)

cap = cv2.VideoCapture(0)

# 제어 변수
last_left_time = 0
scroll_mode = False
scroll_start_y = 0

def get_dist(p1, p2):
    return np.hypot(p1.x - p2.x, p1.y - p2.y)

while cap.isOpened():
    success, img = cap.read()
    if not success: break
    
    img = cv2.flip(img, 1)
    h, w, _ = img.shape
    result = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            lm = hand_landmarks.landmark
            
            # 기준점 설정
            pointer_base = lm[5]  # 검지 뿌리 마디 (가장 안정적)
            thumb_tip = lm[4]     # 엄지 끝 (클릭/조합용)
            index_tip = lm[8]     # 검지 끝 (클릭용)
            middle_tip = lm[12]   # 중지 끝 (우클릭용)
            ring_tip = lm[16]     # 약지 끝 (스크롤용)

            # 1. 마우스 이동 (검지 마디 기준)
            # 마디 기준이라 손가락을 까닥여도 커서가 거의 안 흔들림
            mx = np.interp(pointer_base.x, (0.2, 0.8), (0, screen_w))
            my = np.interp(pointer_base.y, (0.2, 0.8), (0, screen_h))
            pyautogui.moveTo(mx, my, _pause=False)

            # 2. 거리 측정
            d_left = get_dist(thumb_tip, index_tip)   # 좌클릭 (엄지-검지)
            d_right = get_dist(thumb_tip, middle_tip) # 우클릭 (엄지-중지)
            d_scroll = get_dist(thumb_tip, ring_tip)  # 스크롤 (엄지-약지)

            # 3. 기능 구현
            # A. 왼쪽 클릭 & 더블클릭
            if d_left < 0.04:
                curr = time.time()
                if curr - last_left_time > 0.2:
                    if curr - last_left_time < 0.5:
                        pyautogui.doubleClick()
                    else:
                        pyautogui.click()
                    last_left_time = curr
                cv2.putText(img, "LEFT/DOUBLE", (50, 50), 1, 2, (0, 255, 0), 2)

            # B. 오른쪽 클릭
            elif d_right < 0.04:
                pyautogui.rightClick()
                time.sleep(0.3)
                cv2.putText(img, "RIGHT", (50, 50), 1, 2, (0, 0, 255), 2)

            # C. 스크롤 모드 (엄지와 약지를 붙이고 손을 위아래로)
            elif d_scroll < 0.04:
                if not scroll_mode:
                    scroll_mode = True
                    scroll_start_y = pointer_base.y
                else:
                    diff = scroll_start_y - pointer_base.y
                    if abs(diff) > 0.01:
                        pyautogui.scroll(int(diff * 1000))
                        # 스크롤 중에는 기준점을 계속 갱신하여 부드럽게 유지
                        scroll_start_y = pointer_base.y 
                cv2.putText(img, "SCROLLING...", (50, 50), 1, 2, (255, 255, 0), 2)
            else:
                scroll_mode = False

    # 화면 표시
    cv2.imshow("Advanced Controller", cv2.resize(img, (640, 480)))
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()